In [3]:
import numpy as np
import torch
import mne

from pathlib import Path

# EEGBCI 데이터 경로
DATA_PATH = Path(r"C:\bci_data")

# Project 1과 동일한 조건
SUBJECTS = list(range(1, 11))
RUNS = [4, 8, 12]

FMIN = 8.0
FMAX = 30.0

TMIN = 0.0
TMAX = 3.0

print("PyTorch:", torch.__version__)
print("MNE:", mne.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Data path exists:", DATA_PATH.exists())

PyTorch: 2.14.0+cpu
MNE: 1.12.1
CUDA available: False
Data path exists: False


In [5]:
from mne.datasets import eegbci

# 데이터 폴더 생성
DATA_PATH.mkdir(parents=True, exist_ok=True)

# Subject 1의 Motor Imagery runs 다운로드
subjects = [1]
runs = [4, 8, 12]

files = eegbci.load_data(
    subjects=subjects,
    runs=runs,
    path=str(DATA_PATH),
    update_path=False
)

print("Downloaded files:")
for f in files:
    print(f)

Download complete in 01m02s (7.4 MB)
Downloaded files:
C:\bci_data\MNE-eegbci-data\files\eegmmidb\1.0.0\S001\S001R04.edf
C:\bci_data\MNE-eegbci-data\files\eegmmidb\1.0.0\S001\S001R08.edf
C:\bci_data\MNE-eegbci-data\files\eegmmidb\1.0.0\S001\S001R12.edf


In [6]:
raws = []

for f in files:
    raw = mne.io.read_raw_edf(f, preload=True, verbose=False)
    raws.append(raw)

raw = mne.concatenate_raws(raws)

print(raw)
print("Channels:", len(raw.ch_names))
print("Sampling rate:", raw.info["sfreq"])
print("Duration (sec):", raw.times[-1])

<RawEDF | S001R04.edf, 64 x 60000 (375.0 s), ~29.3 MiB, data loaded>
Channels: 64
Sampling rate: 160.0
Duration (sec): 374.99375


In [7]:
# EEGBCI 채널 이름 정리
eegbci.standardize(raw)

# 8–30 Hz band-pass filtering
raw.filter(
    l_freq=FMIN,
    h_freq=FMAX,
    fir_design="firwin",
    verbose=False
)

# Annotation에서 이벤트 추출
events, event_id = mne.events_from_annotations(raw, verbose=False)

print("Event ID:", event_id)
print("Number of events:", len(events))

Event ID: {np.str_('T0'): 1, np.str_('T1'): 2, np.str_('T2'): 3}
Number of events: 90


In [8]:
# T1 / T2만 사용
motor_event_id = {
    "left": event_id["T1"],
    "right": event_id["T2"]
}

epochs = mne.Epochs(
    raw,
    events,
    event_id=motor_event_id,
    tmin=TMIN,
    tmax=TMAX,
    baseline=None,
    preload=True,
    verbose=False
)

X = epochs.get_data()
y = epochs.events[:, -1]

# label을 0 / 1로 변환
y = (y == motor_event_id["right"]).astype(np.int64)

print("Epoch shape:", X.shape)
print("Label shape:", y.shape)
print("Left trials:", np.sum(y == 0))
print("Right trials:", np.sum(y == 1))

Epoch shape: (45, 64, 481)
Label shape: (45,)
Left trials: 23
Right trials: 22


In [9]:
# EEGNet 입력 형태로 변환
# (trials, channels, time) -> (trials, 1, channels, time)

X_eegnet = X[:, np.newaxis, :, :]

print("Original shape:", X.shape)
print("EEGNet shape:", X_eegnet.shape)
print("Labels shape:", y.shape)

Original shape: (45, 64, 481)
EEGNet shape: (45, 1, 64, 481)
Labels shape: (45,)


In [10]:
import torch.nn as nn


class EEGNet(nn.Module):
    def __init__(
        self,
        n_channels=64,
        n_samples=481,
        n_classes=2,
        dropout=0.5
    ):
        super().__init__()

        # 1. 시간 방향 특징 추출
        self.temporal = nn.Sequential(
            nn.Conv2d(
                1,
                8,
                kernel_size=(1, 64),
                padding="same",
                bias=False
            ),
            nn.BatchNorm2d(8)
        )

        # 2. EEG 채널 방향 공간 특징 추출
        self.spatial = nn.Sequential(
            nn.Conv2d(
                8,
                16,
                kernel_size=(n_channels, 1),
                groups=8,
                bias=False
            ),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout)
        )

        # 3. 추가적인 시간 특징 추출
        self.separable = nn.Sequential(
            nn.Conv2d(
                16,
                16,
                kernel_size=(1, 16),
                padding="same",
                groups=16,
                bias=False
            ),
            nn.Conv2d(
                16,
                16,
                kernel_size=(1, 1),
                bias=False
            ),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout)
        )

        # 출력 크기를 자동으로 계산
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            features = self._forward_features(dummy)
            feature_size = features.flatten(1).shape[1]

        self.classifier = nn.Linear(feature_size, n_classes)

    def _forward_features(self, x):
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.separable(x)
        return x

    def forward(self, x):
        x = self._forward_features(x)
        x = x.flatten(1)
        return self.classifier(x)


model = EEGNet()

print(model)

n_params = sum(p.numel() for p in model.parameters())
print("\nTrainable parameters:", n_params)

EEGNet(
  (temporal): Sequential(
    (0): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=same, bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (spatial): Sequential(
    (0): Conv2d(8, 16, kernel_size=(64, 1), stride=(1, 1), groups=8, bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
    (3): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
    (4): Dropout(p=0.5, inplace=False)
  )
  (separable): Sequential(
    (0): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=same, groups=16, bias=False)
    (1): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): ELU(alpha=1.0)
    (4): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
    (5): Dropout(p=0.5, inplace=False)
  )
  (classifier)

C:\Users\User\OneDrive\프로젝트\eeg-motor-imagery-eegnet\eeg-motor-imagery-eegnet\.venv\Lib\site-packages\torch\nn\modules\conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1105.)
  return F.conv2d(


In [11]:
# NumPy -> PyTorch Tensor
X_tensor = torch.tensor(X_eegnet, dtype=torch.float32)

# EEGNet에 데이터 통과
model.eval()

with torch.no_grad():
    output = model(X_tensor)

print("Input shape:", X_tensor.shape)
print("Output shape:", output.shape)
print("First trial output:", output[0])

Input shape: torch.Size([45, 1, 64, 481])
Output shape: torch.Size([45, 2])
First trial output: tensor([-0.0459,  0.0353])


In [12]:
from sklearn.model_selection import train_test_split

# Subject 1을 train / test로 분리
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

print("Train left :", np.sum(y_train == 0))
print("Train right:", np.sum(y_train == 1))
print("Test left  :", np.sum(y_test == 0))
print("Test right :", np.sum(y_test == 1))

Train: (36, 64, 481) (36,)
Test : (9, 64, 481) (9,)
Train left : 18
Train right: 18
Test left  : 5
Test right : 4


In [13]:
# Train 데이터에서만 평균 / 표준편차 계산
mean = X_train.mean(axis=(0, 2), keepdims=True)
std = X_train.std(axis=(0, 2), keepdims=True)

# 0으로 나누는 문제 방지
std[std < 1e-8] = 1.0

# Train 기준으로 Train / Test 모두 표준화
X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

# EEGNet 입력 차원 추가
X_train_scaled = X_train_scaled[:, np.newaxis, :, :]
X_test_scaled = X_test_scaled[:, np.newaxis, :, :]

# PyTorch Tensor로 변환
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("X train:", X_train_tensor.shape)
print("X test :", X_test_tensor.shape)
print("y train:", y_train_tensor.shape)
print("y test :", y_test_tensor.shape)

X train: torch.Size([36, 1, 64, 481])
X test : torch.Size([9, 1, 64, 481])
y train: torch.Size([36])
y test : torch.Size([9])


In [14]:
from torch.utils.data import TensorDataset, DataLoader

# 데이터셋 / 데이터로더
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

# 모델 새로 생성
model = EEGNet()

# 손실함수 / 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

# 간단한 학습
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criterion(outputs, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    train_loss = total_loss / total
    train_acc = correct / total

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.3f}"
        )

Epoch 05 | Loss: 0.6922 | Train Acc: 0.528
Epoch 10 | Loss: 0.6005 | Train Acc: 0.833
Epoch 15 | Loss: 0.6052 | Train Acc: 0.694
Epoch 20 | Loss: 0.5838 | Train Acc: 0.694
Epoch 25 | Loss: 0.5213 | Train Acc: 0.806
Epoch 30 | Loss: 0.5147 | Train Acc: 0.806


In [15]:
model.eval()

correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        preds = outputs.argmax(dim=1)

        correct += (preds == yb).sum().item()
        total += yb.size(0)

        all_preds.extend(preds.tolist())
        all_labels.extend(yb.tolist())

test_acc = correct / total

print(f"Test Accuracy: {test_acc:.3f}")
print("Predicted:", all_preds)
print("Actual   :", all_labels)

Test Accuracy: 0.667
Predicted: [0, 1, 1, 0, 1, 1, 1, 1, 1]
Actual   : [0, 0, 0, 0, 1, 1, 1, 1, 0]
